# Polygon Array Generator (Nazca)

**Author:** Jason P. Beech  
**Date:** 2026-03  
**Affiliation:** Tegenfeldt Lab / Lund University    

This notebook generates an array of repeated **2×2 polygon unit cells** in **Nazca**.

## What the code does
Each unit cell contains four regular polygons:

- triangle
- square
- pentagon
- hexagon

All four shapes use the same **circumradius** and are arranged in a 2×2 grid inside a small cell. That small cell is then repeated in a larger array.

## Main settings
At the top of the code cell, you can change:

- `EXPORT_MODE`
  - `"plot"` → preview only
  - `"gds"` → export GDS only
- `R_um` → circumradius of each polygon
- `space_um` → spacing between shapes and between neighboring cells
- `N`, `M` → number of cells in x and y
- `rot_triangle`, `rot_square`, `rot_pentagon`, `rot_hexagon` → polygon orientations
- `add_outline` → optionally draw a border around the full array

## Geometry notes
- Each polygon is defined by its **circumradius** `R_um`.
- Each small unit cell contains a 2×2 arrangement of shapes.
- The spacing inside a cell and between neighboring cells is controlled by the same parameter, `space_um`.
- The array pitch is therefore:
  - `pitchX_um = cell_w_um + space_um`
  - `pitchY_um = cell_h_um + space_um`

## Output behavior
The notebook supports two output modes:

- `"plot"`: generate a Nazca plot preview
- `"gds"`: export a GDSII file

## Automatic filename
When exporting GDS, the filename is generated automatically from the main design parameters and the current date.

It includes:

- array size: `N x M`
- circumradius: `R_um`
- spacing: `space_um`
- date in `YYYYMMDD` format

For example, a file may be saved as:

```python
polygon_array_50x50_R50um_space20um_20260317.gds

In [7]:
import nazca as nd
import math
from datetime import datetime

# =============================================================================
# Polygon Array Generator (Nazca)
#
# Author: Jason P. Beech
# Date: 2026-03
# Affiliation: Tegenfeldt Lab / Lund University
#
# Description:
# Generates an array of 2x2 unit cells containing a triangle, square,
# pentagon, and hexagon, all with the same circumradius.
# =============================================================================


# =============================================================================
# Output control
# =============================================================================
# Choose one of:
#   "plot" -> preview only
#   "gds"  -> export GDS only
EXPORT_MODE = "gds"


# =============================================================================
# Design parameters (units: µm)
# =============================================================================

R_um = 100.0          # circumradius for all shapes in the small cell
space_um = 20.0      # gap between shapes in a cell and between cells

# Array size: N columns (x direction) by M rows (y direction)
N = 2
M = 2


# =============================================================================
# Polygon rotations (radians)
# =============================================================================
# These are purely for orientation / aesthetics.

rot_triangle = math.pi / 2   # point-up triangle
rot_square = math.pi / 4     # square with flat base
rot_pentagon = math.pi / 2
rot_hexagon = 0.0            # pointy-right hexagon
# Use math.pi / 6 for a flat-top hexagon if preferred.


# =============================================================================
# Optional outline
# =============================================================================

add_outline = False


# =============================================================================
# Auto-generated filename
# =============================================================================

date_tag = datetime.now().strftime("%Y%m%d")

GDS_FILENAME = (
    f"polygon_array_"
    f"{N}x{M}_"
    f"R{R_um:g}um_"
    f"space{space_um:g}um_"
    f"{date_tag}.gds"
)


# =============================================================================
# Helper functions
# =============================================================================

def regular_ngon(cx, cy, n, R, rotation=0.0, layer=1):
    """
    Place a regular n-gon centered at (cx, cy) with circumradius R.

    Parameters
    ----------
    cx, cy : float
        Center coordinates of the polygon.
    n : int
        Number of polygon sides.
    R : float
        Circumradius of the polygon in µm.
    rotation : float
        Rotation angle in radians.
    layer : int
        GDS layer number.
    """
    pts = []
    for k in range(n):
        t = rotation + 2.0 * math.pi * k / n
        x = cx + R * math.cos(t)
        y = cy + R * math.sin(t)
        pts.append((x, y))

    pts.append(pts[0])  # close polygon
    nd.Polygon(pts, layer=layer).put(0, 0)


# =============================================================================
# Small-cell geometry (2x2 grid of shapes)
# =============================================================================
# Each shape fits in a box of size 2R x 2R.
# The small cell contains 2 columns x 2 rows with 'space_um' gaps.

shape_diam = 2.0 * R_um
cell_w_um = 2 * shape_diam + space_um
cell_h_um = 2 * shape_diam + space_um


def cell_shape_centers(x0, y0):
    """
    Return the centers of the four shapes inside one small cell.

    Parameters
    ----------
    x0, y0 : float
        Bottom-left corner of the small cell.

    Returns
    -------
    dict
        Center coordinates for triangle, square, pentagon, and hexagon.
    """
    cx_left = x0 + R_um
    cx_right = x0 + R_um + shape_diam + space_um
    cy_bottom = y0 + R_um
    cy_top = y0 + R_um + shape_diam + space_um

    return {
        "triangle": (cx_left,  cy_bottom),
        "square":   (cx_right, cy_bottom),
        "pentagon": (cx_left,  cy_top),
        "hexagon":  (cx_right, cy_top),
    }


def place_small_cell(x0, y0, layer=1):
    """
    Place one 2x2 small cell of polygons.
    """
    centers = cell_shape_centers(x0, y0)

    regular_ngon(*centers["triangle"], n=3, R=R_um, rotation=rot_triangle, layer=layer)
    regular_ngon(*centers["square"],   n=4, R=R_um, rotation=rot_square,   layer=layer)
    regular_ngon(*centers["pentagon"], n=5, R=R_um, rotation=rot_pentagon, layer=layer)
    regular_ngon(*centers["hexagon"],  n=6, R=R_um, rotation=rot_hexagon,  layer=layer)


# =============================================================================
# Array placement
# =============================================================================

pitchX_um = cell_w_um + space_um
pitchY_um = cell_h_um + space_um

print(f"Small cell size: {cell_w_um:.1f} µm × {cell_h_um:.1f} µm")
print(f"Array: {N} × {M}, pitchX = {pitchX_um:.1f} µm, pitchY = {pitchY_um:.1f} µm")
print(f"Export mode: {EXPORT_MODE}")
print(f"GDS filename: {GDS_FILENAME}")

for j in range(M):
    for i in range(N):
        x = i * pitchX_um
        y = j * pitchY_um
        place_small_cell(x, y, layer=1)


# =============================================================================
# Optional outline around the full array
# =============================================================================

if add_outline:
    array_w_um = (N - 1) * pitchX_um + cell_w_um
    array_h_um = (M - 1) * pitchY_um + cell_h_um
    nd.Polygon(
        [(0, 0), (array_w_um, 0), (array_w_um, array_h_um), (0, array_h_um), (0, 0)],
        layer=1
    ).put(0, 0)


# =============================================================================
# Export / preview
# =============================================================================

mode = EXPORT_MODE.lower()

if mode == "gds":
    nd.export_gds(filename=GDS_FILENAME)
    print(f"GDS exported: {GDS_FILENAME}")

elif mode == "plot":
    nd.export_plt()
    print("Plot exported.")

else:
    raise ValueError("EXPORT_MODE must be one of: 'plot' or 'gds'")

Small cell size: 420.0 µm × 420.0 µm
Array: 2 × 2, pitchX = 440.0 µm, pitchY = 440.0 µm
Export mode: gds
GDS filename: polygon_array_2x2_R100um_space20um_20260317.gds
Starting layout export...
...gds generation
...Wrote file './polygon_array_2x2_R100um_space20um_20260317.gds'


GDS exported: polygon_array_2x2_R100um_space20um_20260317.gds
